# PGM E0.R — Relational Component Occurrence & Nested Spatial Qualification

Source-locked execution of preregistered Issue #80. R1 runs on all six frozen E0.2 representations. R2 is implemented in the locked source but runs only if the registered R1 joint gate passes. Official Train supervision and PublicTest developmental labels are read only after all R1 occurrence representations are frozen. PrivateTest is forbidden.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
REPO_BRANCH = 'research/pixel-relational-motif-e0r'
SOURCE_SHA = '671e3c2f69607778f08a923026e76744561e13b2'
EXPECTED_TRAIN_SHA256 = 'deb82c4b4e01b90776a718c34934666b0bdde6696ca1d0149f8fe807a8ff4ba8'
EXPECTED_PUBLIC_SHA256 = '412036d077c6ec203047b2935ab14bc858d8136ee26e8db3e23023f1fc9dee08'
EXPECTED_DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
EXPECTED_E02_SUMMARY_SHA256 = 'edd7c0b01f3696b941f34043a3758d2e5e9af9da595b994bfd257b8a61789a1e'
EXPECTED_E02_RESULTS_SHA256 = 'cd41e6793ef3451ee1d11b1c32e0178026c45532e6386fe2800164d34e6b4cad'
EXPECTED_CONTROL_SHA256 = {
    42: 'ace39fc9f8bac39523080002acab547c4c65cf6015fda8b09101003238d5c1e6',
    43: '4795b0380ea25134878f80202cbc9ab737d1096091728f25af87819590bb4ea9',
    44: 'c2d881b9806ee927c3a446a3f79fc7ea15faf1cd2442f39a16283a948af94a07',
    45: '2b03c9da78a970cdf52f86b8211761e1eb7d2436a780f19a223038de70c7bef9',
    46: '0ddea5e79dbe512e3af91f5d2778082832b6a12fe7f63192ce9ecc7a15e8d415',
}
FER_ROOT = Path('/kaggle/input/fer13-split/fer13-split')
TRAIN_CSV = FER_ROOT / 'train.csv'
PUBLIC_CSV = FER_ROOT / 'val.csv'
DICTIONARY_NPZ = Path('/kaggle/input/pgm-e01-v533-dictionary/e01_dictionary.npz')
E02_ROOT = Path('/kaggle/input/pgm-e02-v535-artifacts')
E02_SUMMARY = E02_ROOT / 'e02_summary.json'
E02_RESULTS = E02_ROOT / 'e02_results.npz'
CONTROL_PATHS = {seed: E02_ROOT / f'e02_control_seed{seed}.npz' for seed in EXPECTED_CONTROL_SHA256}
OUTPUT_DIR = Path('/kaggle/working/outputs/pixel_relational_motif_e0r')
PACKAGE_RELATIVE = Path('research/pixel_relational_motif_e0')
RUN_TESTS = True
RUN_E0R = True


In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, time
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'FER2013_Graph_E0R'
if PROJECT.exists(): shutil.rmtree(PROJECT)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_URL,str(PROJECT)], check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_SHA], check=True)
head = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'], text=True).strip()
if head != SOURCE_SHA: raise RuntimeError(f'source lock mismatch: {head} != {SOURCE_SHA}')
subprocess.run(['git','-C',str(PROJECT),'diff','--quiet'], check=True)
subprocess.run(['git','-C',str(PROJECT),'diff','--cached','--quiet'], check=True)
PACKAGE = PROJECT / PACKAGE_RELATIVE
PACKAGE_SRC = PACKAGE / 'src'
sys.path.insert(0, str(PACKAGE_SRC))
import pixel_relational_motif_e0
imported = Path(pixel_relational_motif_e0.__file__).resolve()
if PACKAGE_SRC.resolve() not in imported.parents: raise RuntimeError(f'import isolation violation: {imported}')
print('Expected scientific source SHA:', SOURCE_SHA)
print('Actual checked-out source SHA:', head)
print('Source lock PASS')


In [ ]:
if RUN_TESTS:
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PACKAGE_SRC) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    result = subprocess.run([sys.executable,'-m','pytest',str(PACKAGE/'tests'),'-q'], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if result.returncode != 0: raise RuntimeError(f'pre-data pytest failed: {result.returncode}')
    print('Pre-data regression tests PASS')


In [ ]:
import numpy as np, scipy, sklearn
environment = {
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'scipy': scipy.__version__,
    'sklearn': sklearn.__version__,
    'source_sha': SOURCE_SHA,
}
print('Environment:', json.dumps(environment, indent=2))


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

if not TRAIN_CSV.is_file(): raise FileNotFoundError(TRAIN_CSV)
if not PUBLIC_CSV.is_file(): raise FileNotFoundError(PUBLIC_CSV)
if not DICTIONARY_NPZ.is_file(): raise FileNotFoundError(DICTIONARY_NPZ)
if not E02_SUMMARY.is_file(): raise FileNotFoundError(E02_SUMMARY)
if not E02_RESULTS.is_file(): raise FileNotFoundError(E02_RESULTS)
for path in CONTROL_PATHS.values():
    if not path.is_file(): raise FileNotFoundError(path)
if sha256(TRAIN_CSV) != EXPECTED_TRAIN_SHA256: raise RuntimeError('Train SHA mismatch')
if sha256(PUBLIC_CSV) != EXPECTED_PUBLIC_SHA256: raise RuntimeError('Public SHA mismatch')
if sha256(DICTIONARY_NPZ) != EXPECTED_DICTIONARY_SHA256: raise RuntimeError('dictionary SHA mismatch')
if sha256(E02_SUMMARY) != EXPECTED_E02_SUMMARY_SHA256: raise RuntimeError('E0.2 summary SHA mismatch')
if sha256(E02_RESULTS) != EXPECTED_E02_RESULTS_SHA256: raise RuntimeError('E0.2 results SHA mismatch')
for seed, path in CONTROL_PATHS.items():
    if sha256(path) != EXPECTED_CONTROL_SHA256[seed]: raise RuntimeError(f'control {seed} SHA mismatch')
print('Resolved immutable inputs:')
for role, path in [('Train',TRAIN_CSV),('PublicTest',PUBLIC_CSV),('actual-v533',DICTIONARY_NPZ),('E0.2-summary',E02_SUMMARY),('E0.2-results',E02_RESULTS),*[('control-'+str(seed),path) for seed,path in CONTROL_PATHS.items()]]:
    print(f'  {role}: {path} sha256={sha256(path)}')
print('No PrivateTest path is configured or inspected.')


In [ ]:
import contextlib, threading
@contextlib.contextmanager
def heartbeat(label, interval_seconds=180):
    stop = threading.Event(); started = time.time()
    def worker():
        while not stop.wait(interval_seconds): print(f'[heartbeat] {label}: {(time.time()-started)/60:.1f} min', flush=True)
    thread = threading.Thread(target=worker, daemon=True); thread.start()
    try: yield
    finally:
        stop.set(); thread.join(timeout=1); print(f'[heartbeat] {label}: complete {(time.time()-started)/60:.1f} min', flush=True)


In [ ]:
from pixel_relational_motif_e0.e0r_runner import run_e0r
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2, sort_keys=True), encoding='utf-8')
if RUN_E0R:
    with heartbeat('PGM E0.R sequential scientific execution'):
        summary = run_e0r(TRAIN_CSV, PUBLIC_CSV, DICTIONARY_NPZ, CONTROL_PATHS, E02_SUMMARY, E02_RESULTS, OUTPUT_DIR)
    print(json.dumps({
        'r1_verdict': summary['r1']['comparison']['registered_verdict'],
        'r1_comparison': summary['r1']['comparison'],
        'r2': summary['r2'],
        'private_test_read': summary['private_test_read'],
    }, indent=2))


In [ ]:
required = {'environment.json','e0r_summary.json','e0r_artifact_manifest.json','e0r_occurrences_actual.npz','e0r_r1_occurrence_diagnostics.npz','e0r_r1_results.npz'}
present = {path.name for path in OUTPUT_DIR.iterdir() if path.is_file()}
missing = sorted(required - present)
if missing: raise RuntimeError(f'missing E0.R artifacts: {missing}')
if not ({'e0r_r2_status.json','e0r_r2_results.npz'} & present): raise RuntimeError('missing sequential R2 status/results artifact')
final_manifest = {path.name: {'sha256': sha256(path), 'bytes': path.stat().st_size} for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file()}
print('Final artifact inventory:', json.dumps(final_manifest, indent=2))
print('PGM E0.R execution complete. E0.1/E0.1b/E0.2 remain unchanged; M0 was not executed.')
